Name: Dev Patel 

Course: DS4400 Data Mining and Machine Learning 1

Prof: Silvio Amir

University: Northeastern University

Problem 2: Gradient Descent for Logistic Regression

1. Adapt gradient descent from HW2 for logistic regression. Take 3 learning rates and report cross-entropy loss after 10, 50, 100 iterations.
2. At 100 iterations, report accuracy, precision, recall, F1 for 3 learning rates. Compare with sklearn package metrics on train and test sets.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# Load SPAMBASE dataset
col_names = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our',
    'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail',
    'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses',
    'word_freq_free', 'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money', 'word_freq_hp',
    'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs',
    'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85',
    'word_freq_technology', 'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project', 'word_freq_re',
    'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$', 'char_freq_#',
    'capital_run_length_average', 'capital_run_length_longest', 'capital_run_length_total',
    'spam'
]

df = pd.read_csv('spambase.data', header=None, names=col_names)
X = df.drop('spam', axis=1)
y = df['spam']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Standardize features for gradient descent stability
train_mean = X_train.mean()
train_std = X_train.std()
train_std[train_std == 0] = 1

X_train_scaled = ((X_train - train_mean) / train_std).values.astype(float)
X_test_scaled = ((X_test - train_mean) / train_std).values.astype(float)
y_train_np = y_train.values.astype(float)
y_test_np = y_test.values.astype(float)

print(f"X_train: {X_train_scaled.shape}, y_train: {y_train_np.shape}")
print(f"X_test: {X_test_scaled.shape}, y_test: {y_test_np.shape}")

X_train: (3450, 57), y_train: (3450,)
X_test: (1151, 57), y_test: (1151,)


In [3]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def cross_entropy_loss(X, y, theta):
    """
    Cross-entropy loss: J(θ) = -(1/n) * Σ [y*log(h) + (1-y)*log(1-h)]
    where h = σ(Xθ)
    """
    n = len(y)
    X_design = np.column_stack([np.ones(n), X])
    h = sigmoid(X_design @ theta)
    eps = 1e-15
    h = np.clip(h, eps, 1 - eps)
    return -(1 / n) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h))

def logistic_gradient_descent(X, y, alpha, n_iters):
    """
    Gradient descent for logistic regression.
    
    Gradient: ∇J = (1/n) * X^T (σ(Xθ) - y)
    Update: θ := θ - α * ∇J
    
    X: (n, p) feature matrix (no intercept column; added inside)
    Returns: theta (p+1,), dict of losses at snapshot iterations
    """
    n = len(y)
    X_design = np.column_stack([np.ones(n), X])
    p = X_design.shape[1]
    theta = np.zeros(p)
    
    losses = {}
    for i in range(1, n_iters + 1):
        h = sigmoid(X_design @ theta)
        gradient = (1 / n) * (X_design.T @ (h - y))
        theta = theta - alpha * gradient
        
        if i in [10, 50, 100]:
            losses[i] = cross_entropy_loss(X, y, theta)
    
    return theta, losses

def predict_gd(X, theta, threshold=0.5):
    X_design = np.column_stack([np.ones(len(X)), X])
    proba = sigmoid(X_design @ theta)
    return (proba >= threshold).astype(int)

In [4]:
alphas = [0.01, 0.1, 0.5]
iterations_list = [10, 50, 100]

# Report cross-entropy loss at 10, 50, 100 iterations
loss_rows = []
for alpha in alphas:
    theta, losses = logistic_gradient_descent(X_train_scaled, y_train_np, alpha, 100)
    for n_iter in iterations_list:
        loss_rows.append({'α': alpha, 'Iterations': n_iter, 'Cross-Entropy Loss': round(losses[n_iter], 6)})

loss_df = pd.DataFrame(loss_rows)
print("Cross-Entropy Loss at Different Iterations:\n")
loss_df

Cross-Entropy Loss at Different Iterations:



,α,Iterations,Cross-Entropy Loss
0,0.01,10,0.651178
1,0.01,50,0.541647
2,0.01,100,0.468763
3,0.10,10,0.465407
4,0.10,50,0.324140
5,0.10,100,0.289089
6,0.50,10,0.320266
7,0.50,50,0.258901
8,0.50,100,0.243783


In [5]:
# At 100 iterations, report accuracy, precision, recall, F1
metric_rows = []
for alpha in alphas:
    theta, _ = logistic_gradient_descent(X_train_scaled, y_train_np, alpha, 100)
    
    for label, X_eval, y_eval in [('Train', X_train_scaled, y_train_np), ('Test', X_test_scaled, y_test_np)]:
        y_pred = predict_gd(X_eval, theta)
        metric_rows.append({
            'α': alpha,
            'Set': label,
            'Accuracy': round(accuracy_score(y_eval, y_pred), 4),
            'Precision': round(precision_score(y_eval, y_pred, zero_division=0), 4),
            'Recall': round(recall_score(y_eval, y_pred, zero_division=0), 4),
            'F1': round(f1_score(y_eval, y_pred, zero_division=0), 4)
        })

metric_df = pd.DataFrame(metric_rows)
print("Gradient Descent Metrics at 100 Iterations:\n")
metric_df

Gradient Descent Metrics at 100 Iterations:



,α,Set,Accuracy,Precision,Recall,F1
0,0.01,Train,0.8977,0.8734,0.8610,0.8671
1,0.01,Test,0.9036,0.9009,0.8611,0.8805
2,0.10,Train,0.9078,0.9113,0.8445,0.8766
3,0.10,Test,0.9062,0.9238,0.8421,0.8811
4,0.50,Train,0.9183,0.9238,0.8602,0.8909
5,0.50,Test,0.9157,0.9395,0.8505,0.8928


In [6]:
# Compare with sklearn LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sk = scaler.fit_transform(X_train)
X_test_sk = scaler.transform(X_test)

sk_model = LogisticRegression(max_iter=10000, random_state=42)
sk_model.fit(X_train_sk, y_train)

sk_rows = []
for label, X_eval, y_eval in [('Train', X_train_sk, y_train), ('Test', X_test_sk, y_test)]:
    y_pred_sk = sk_model.predict(X_eval)
    sk_rows.append({
        'Model': 'sklearn',
        'Set': label,
        'Accuracy': round(accuracy_score(y_eval, y_pred_sk), 4),
        'Precision': round(precision_score(y_eval, y_pred_sk, zero_division=0), 4),
        'Recall': round(recall_score(y_eval, y_pred_sk, zero_division=0), 4),
        'F1': round(f1_score(y_eval, y_pred_sk, zero_division=0), 4)
    })

sk_df = pd.DataFrame(sk_rows)
print("sklearn LogisticRegression Metrics:\n")
sk_df

sklearn LogisticRegression Metrics:



,Model,Set,Accuracy,Precision,Recall,F1
0,sklearn,Train,0.9258,0.9240,0.8812,0.9021
1,sklearn,Test,0.9227,0.9406,0.8674,0.9025


**Observations:**

- **Small learning rate (α = 0.01):** Convergence is slow. The cross-entropy loss decreases gradually but has not fully converged by 100 iterations. The metrics at 100 iterations may be noticeably worse than sklearn because the model has not reached the optimum.

- **Moderate learning rate (α = 0.1):** Good balance between convergence speed and stability. The loss drops significantly by 50 iterations, and by 100 iterations the metrics approach the sklearn solution.

- **Larger learning rate (α = 0.5):** Converges fastest among the three rates. The loss drops quickly and the metrics at 100 iterations are close to the sklearn solution. However, if the rate were too large, the algorithm could oscillate or diverge.

- **Comparison with sklearn:** The sklearn LogisticRegression uses L-BFGS or other advanced optimizers, which converge much faster than basic gradient descent. With enough iterations and a good learning rate, gradient descent approaches the same solution, but sklearn typically achieves it in fewer effective iterations.

- **Feature standardization:** Just as in HW2, standardizing features is critical for gradient descent to work well. Without scaling, features with different magnitudes would cause the loss surface to be elongated, making convergence difficult.